# 03 - Train GFlowNet on Colab A100
Transformer policy, Trajectory Balance, PyTorch AMP, checkpoint save, reload test, and alpha-pool generation.

In [ ]:
from pathlib import Path
import os
if not Path('configs/training_config.yaml').exists():
    repo=os.environ.get('ALPHAMINING_REPO_URL','https://github.com/YOUR_GITHUB_USERNAME/AlphaMining-GFlowNet-AlphaEval.git')
    if 'YOUR_GITHUB_USERNAME' in repo: raise RuntimeError('Set ALPHAMINING_REPO_URL')
    !git clone $repo
    %cd AlphaMining-GFlowNet-AlphaEval

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import torch
print('PyTorch:',torch.__version__)
print('CUDA status:',torch.cuda.is_available(),'CUDA runtime:',torch.version.cuda)
assert torch.cuda.is_available(),'Enable a Colab A100 GPU runtime'
p=torch.cuda.get_device_properties(0)
print('GPU model:',p.name)
print('GPU memory (GB):',round(p.total_memory/1024**3,2))
assert 'A100' in p.name.upper(),f'A100 required; detected {p.name}'

In [ ]:
import yaml
config=yaml.safe_load(open('configs/training_config.yaml',encoding='utf-8'))
config

In [ ]:
from src.gflownet.run_training import run
run('configs/training_config.yaml', require_a100=True, pool_size=100)

## Checkpoint reload test

In [ ]:
import pandas as pd
from src.gflownet import GFlowNetTrainer, RewardEvaluator
daily=pd.read_pickle(config['dataset']['output'])
evaluator=RewardEvaluator(daily,**config['reward'])
loaded=GFlowNetTrainer.load_checkpoint('checkpoints/gflownet_best.pt',evaluator,device='cuda')
expression,_,tokens=loaded.sample_trajectory(greedy=True)
print('Reload successful:',expression)
print('Tokens:',tokens)
assert Path('results/alpha_pool.csv').exists()